In [2]:
import tensorflow as tf

In [21]:
import numpy as np
from tensorflow import keras
import pathlib

Aqui nós vamos definir o modelo, atribuir toda a camada de neuronios à var mylayer

In [13]:
my_layer = keras.layers.Dense(units=1, input_shape=[1])
#aqui nos definimos o modelo
modelo = tf.keras.Sequential([my_layer])

c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


aqui vamos usar a funcao de otimização 'sgd = descida de gradiente estocastico' e para o erro da funcao o erro quadrado medio

In [15]:
modelo.compile(optimizer='sgd', loss = 'mean_squared_error')

Aqui nos vamos definir o conjunto de dados para aprendizagem(xs significa as entradas e ys significa as saidas )

In [16]:
xs= np.array([4.0, 2.0, -1.0, 2.0, 0.0, 3.0], dtype=float)
ys = np.array([7.0, 3.0, -3.0, 1.0, -1.0, 5.0], dtype=float)

In [17]:
#treinamos o nosso modelo com 500 epochs
modelo.fit(xs, ys, epochs=500)

Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - loss: 31.8765
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 25.0000
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 19.7157
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 15.6532
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 12.5286
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 10.1237
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 8.2712
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 6.8428
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 5.7399
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 4.8868
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 4.2257
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.7119
Epoch 13/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 3.3114
Epoch 14/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 2.9978
Epoch 15/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.7511
Epoch 16/500


In [ ]:
#os valores dos parametros neurais (w e b) determinados pela aprendizagem
print("Aqui está o que eu aprendi sobre w e b")
print(my_layer.get_weights())

#o modelo aprendeu a funcao y = 1.9557927 x -1.251301x

Aqui está o que eu aprendi sobre w e b
[array([[1.9557927]], dtype=float32), array([-1.251301], dtype=float32)]


In [20]:
#vamos prever o valor de y quando x= 10
print(modelo.predict(np.array([10.0])))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
[[18.306625]]


In [22]:
#exportando o modelo para o formato SavedModel
saved_dir = 'saved_model/1'#diretorio
modelo.export(saved_dir)
print(f'Modelo salvo em: {pathlib.Path(saved_dir).resolve()}')

INFO:tensorflow:Assets written to: saved_model/1\assets


INFO:tensorflow:Assets written to: saved_model/1\assets


Saved artifact at 'saved_model/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor_3')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2702132461008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2702132461200: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo salvo em: C:\Users\iango\OneDrive\Desktop\tinyml\saved_model\1


Aqui vamos converter para TFLite 


In [23]:
#configurando o TensorFlow Lite Converter
converter = tf.lite.TFLiteConverter.from_saved_model(saved_dir) #criar o conversor
converter.experimental_enable_resource_variables = True #dobra as variaveis em constantes
tflite_model = converter.convert() #executa a conversão

In [24]:
tflite_path = pathlib.Path('model_from_savedmodel.tflite') #caminho de saida
tflite_path.write_bytes(tflite_model)
print(f'Modelo TFLite salvo em: {tflite_path.resolve()}')

Modelo TFLite salvo em: C:\Users\iango\OneDrive\Desktop\tinyml\model_from_savedmodel.tflite


In [28]:
#inicializar interpreter TFLite
interpreter = tf.lite.Interpreter(model_path=str(tflite_path)) #carrega o modelo
interpreter.allocate_tensors() #aloca memororia para  tensores

c:\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [26]:
#recuperar detalhes dos tensores de entrada e saída
input_details = interpreter.get_input_details() #metadados sobre tensor de entrada
output_details = interpreter.get_output_details() #metadados sobre tensor de saída
print(input_details)
print(output_details)

[{'name': 'serving_default_keras_tensor_3:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall_1:0', 'index': 3, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [30]:
#executando a inferencia
to_predict = np.array([[10.0]], dtype=np.float32) #dados de entrada formatados
interpreter.set_tensor(input_details[0]['index'], to_predict) #definir tensor de entrada
interpreter.invoke() #executar a inferencia
tflite_results = interpreter.get_tensor(output_details[0]['index']) #obter resultados
print("TfLite result for x=10.0:", tflite_results)


TfLite result for x=10.0: [[18.306625]]
